# 04 - Fallacy classifier
Fine-tunes DistilRoBERTa on `tasksource/logical-fallacy`. **Use a GPU runtime** (Colab: Runtime > Change runtime type > T4). Training takes a few minutes.

In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules:  # Colab: clone the repo and install dependencies
    if not os.path.isdir("fact-and-logic"):
        subprocess.run(["git", "clone", "https://github.com/Anishkumar25/fact-and-logic.git"], check=True)
    os.chdir("fact-and-logic")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
from src.fallacy import train

res = train(epochs=4)
res

## Per-class results on the test split
Macro-F1 matters more than accuracy here, because some fallacy types are rare.

In [ ]:
from datasets import load_dataset
from sklearn.metrics import classification_report
from src.fallacy import FallacyDetector, LABEL, TEXT

det = FallacyDetector()
test = load_dataset('tasksource/logical-fallacy')['test']
y = [str(x) for x in test[LABEL]]
p = [det.predict(t, top=1)[0]['label'] for t in test[TEXT]]
print(classification_report(y, p, zero_division=0))

In [ ]:
for t in ['You cannot trust his research, he never went to a good university.',
          'Everyone is buying it, so it must be good.',
          'If we allow this, soon everything will collapse.']:
    print(t, '->', det.predict(t))

## Publish to the Hub (free)
The trained model lives only on this Colab session and disappears when it ends, so push it to your Hugging Face account. Create a token with **write** access at huggingface.co/settings/tokens, run `login()`, paste the token, and fill in your username. The deployed app loads the model from the `FALLACY_MODEL` variable.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('Paste your HF write token: '))

repo = 'Anish-25/fact-and-logic'  # your Hugging Face username / repo
det.m.push_to_hub(repo)
det.tok.push_to_hub(repo)

## Choose a confidence cutoff
This dataset has no 'no fallacy' class, so the model names a fallacy even for a plain factual sentence. The table shows, for each cutoff, how much of the real fallacy test set survives, how accurate those guesses are, and how often an ordinary scientific claim still gets flagged. Pick one and set it as `FALLACY_MIN_PROB` (default 0.5). This cell loads the published model, so it also works in a fresh session.

In [ ]:
import numpy as np
from datasets import load_dataset
from src import data
from src.fallacy import FallacyDetector, LABEL, TEXT

det = FallacyDetector('Anish-25/fact-and-logic')
test = load_dataset('tasksource/logical-fallacy')['test']
top = [det.predict(t, top=1)[0] for t in test[TEXT]]
conf = np.array([x['prob'] for x in top])
ok = np.array([x['label'] == str(y) for x, y in zip(top, test[LABEL])])
plain = np.array([det.predict(c, top=1)[0]['prob'] for c in data.claims('validation').claim.head(150)])
for t in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    m = conf >= t
    print(f'cutoff {t}: keeps {m.mean():.0%} of fallacy examples, {ok[m].mean() if m.any() else 0:.0%} of those correct, flags {np.mean(plain >= t):.0%} of ordinary claims')